# 어텐션 헤드의 합성 (Composition) - 세 가지 합성 방식

- Tutorial ID: `tut-3-1`
- Tutorial: 어텐션 헤드의 합성 (Composition)
- Section ID: `s3-1-1`
- Section: 세 가지 합성 방식


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) 위치 정보가 벡터 회전/편향으로 attention score에 들어가는 방식 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 55)
print("어텐션 헤드 합성 (Composition) 분석")
print("=" * 55)

d_model = 8
d_head = 4
n_heads_per_layer = 2

np.random.seed(42)

In [ ]:
# 레이어 0 헤드 (1층)

In [ ]:
W_Q0 = [np.random.randn(d_model, d_head) * 0.3 for _ in range(n_heads_per_layer)]
W_K0 = [np.random.randn(d_model, d_head) * 0.3 for _ in range(n_heads_per_layer)]
W_V0 = [np.random.randn(d_model, d_head) * 0.3 for _ in range(n_heads_per_layer)]
W_O0 = [np.random.randn(d_head, d_model) * 0.3 for _ in range(n_heads_per_layer)]

In [ ]:
# 레이어 1 헤드 (2층)

In [ ]:
W_Q1 = [np.random.randn(d_model, d_head) * 0.3 for _ in range(n_heads_per_layer)]
W_K1 = [np.random.randn(d_model, d_head) * 0.3 for _ in range(n_heads_per_layer)]
W_V1 = [np.random.randn(d_model, d_head) * 0.3 for _ in range(n_heads_per_layer)]
W_O1 = [np.random.randn(d_head, d_model) * 0.3 for _ in range(n_heads_per_layer)]

In [ ]:
# 합성 강도 측정

In [ ]:
def frobenius_composition(A, B):
    """합성 강도: ||A·B||_F / (||A||_F · ||B||_F)"""
    AB = A @ B
    return np.linalg.norm(AB) / (np.linalg.norm(A) * np.linalg.norm(B) + 1e-8)

print("
합성 강도 행렬 (레이어1헤드 × 레이어0헤드):")
print("(랜덤 행렬 기대값 ≈ 1/√d_head)")

print("
Q-합성 (W_QK^1 @ W_OV^0의 프로베니우스 노름비):")
print("     Head0-L0  Head1-L0")
for j in range(n_heads_per_layer):
        W_QK_1j = W_Q1[j] @ W_K1[j].T  # 레이어1, 헤드j의 QK 행렬
    row = []
        for i in range(n_heads_per_layer):
                W_OV_0i = W_V0[i] @ W_O0[i]  # 레이어0, 헤드i의 OV 행렬
        comp = frobenius_composition(W_QK_1j, W_OV_0i)
        row.append(f"{comp:.4f}")
    print(f"  Head{j}-L1: {' '.join(row)}")

print("
K-합성 (W_QK^1.T @ W_OV^0의 프로베니우스 노름비):")
print("     Head0-L0  Head1-L0")
for j in range(n_heads_per_layer):
        W_QK_1j = W_Q1[j] @ W_K1[j].T
    row = []
        for i in range(n_heads_per_layer):
                W_OV_0i = W_V0[i] @ W_O0[i]
        comp = frobenius_composition(W_QK_1j.T, W_OV_0i)
        row.append(f"{comp:.4f}")
    print(f"  Head{j}-L1: {' '.join(row)}")

print("
V-합성 (W_OV^1 @ W_OV^0의 프로베니우스 노름비):")
print("     Head0-L0  Head1-L0")
for j in range(n_heads_per_layer):
        W_OV_1j = W_V1[j] @ W_O1[j]  # 레이어1, 헤드j의 OV 행렬
    row = []
        for i in range(n_heads_per_layer):
                W_OV_0i = W_V0[i] @ W_O0[i]
        comp = frobenius_composition(W_OV_1j, W_OV_0i)
        row.append(f"{comp:.4f}")
    print(f"  Head{j}-L1: {' '.join(row)}")

# 기대 기준값
expected_random = 1 / np.sqrt(d_head)
print(f"
랜덤 행렬 기대값 ≈ 1/√{d_head} = {expected_random:.4f}")
print("이 값보다 현저히 크면 실제 합성이 일어나고 있음!")

In [ ]:
# 합성의 의미

In [ ]:
print("
" + "=" * 55)
print("세 가지 합성의 의미")
print("=" * 55)
print("""
Q-합성: 레이어1 헤드가 '무엇을 찾을지'가 레이어0 출력에 의존
  → 레이어0이 특정 패턴을 주석(annotate)하면
    레이어1이 그 주석을 이용해 더 복잡한 패턴 탐색

K-합성: 레이어1 헤드가 '어디서 찾을지'가 레이어0 출력에 의존
  → 레이어0이 특정 토큰을 표시(mark)하면
    레이어1이 그 표시를 키로 사용

V-합성: 레이어1이 이동시키는 '정보'가 레이어0 출력
  → 레이어0이 처리한 정보를
    레이어1이 다시 이동 (가상 어텐션 헤드)

인덕션 헤드 = K-합성의 핵심 사례:
  레이어0: 이전 토큰 헤드 (each token attends to previous)
  레이어1: 레이어0의 K-composition → 패턴 매칭 헤드
""")